# Manning's Roughness Map Builder 

Burn Manning values on top of each other
Combines three layers, in priority order (highest wins):

1. **Main river bed** (`river_reavhes_polygons.shp`, `manning_n` field) — highest priority
2. **River branches** (`river_mask.tif`, binary mask) — single fixed value
3. **Existing baseline Manning's n raster** — fills everywhere else

**To reuse for a different baseline manning file:** only change `EXISTING_MANNING_TIF`
and `OUTPUT_MANNING_TIF` in the config cell below, then run all cells.

In [ ]:
import os
import csv
import numpy as np
import rasterio
from rasterio.warp import reproject, calculate_default_transform, Resampling
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import Point

## Configure me

In [ ]:
# ============================================================
# Change these for a different baseline Manning file
# ============================================================
EXISTING_MANNING_TIF = r"path/to/manninginput.tif"
OUTPUT_MANNING_TIF = r"path/to/manningoutput.tif"

# ============================================================
# Fixed inputs -- reused across every scenario, shouldn't need to change
# ============================================================
BRANCH_NETWORK_TIF = r"path/to/riverorbranchnetwork.tif"
BRANCH_MANNING_VALUE = 0.033

BUFFER_NETWORK_SHP = r"path/to/river_reaches.shp"
BUFFER_MANNING_FIELD = "manning_n" #this is the name of the column in the attribute table of the shapefile that contains the Manning's n values for each reach
GAP_CLOSE_BUFFER_M = 1.0  # small buffer to close slivers/gaps between adjacent river-bed segments # you can choose

TARGET_CRS = "EPSG:32632"  # <-- set to YOUR project's projected CRS (metres); everything is aligned to it

## Step 1 — load + reproject the existing manning raster (defines the master grid)

No CORINE reclassification here since this file already contains Manning's n values directly -- we just need it on the correct CRS and a fine enough resolution.

**Resampling is nearest-neighbour, not bilinear.** Every output cell keeps an exact n value from
the source (one of the lookup-table values, or a river value) -- never an interpolated value
between two classes. With bilinear, cells along class boundaries would get blended values that
belong to no land-use class, so a land-use change could no longer be traced to a specific
roughness change.

In [ ]:
def reproject_array_to_new_crs(array, src_transform, src_crs, dst_crs, resampling, target_resolution=None):
    """Reproject an in-memory array to a new CRS. If target_resolution is given
    (meters), the output grid is forced to that pixel size; otherwise GDAL picks
    one automatically based on the source resolution.
    This defines the MASTER grid that every other layer gets aligned to."""
    height, width = array.shape
    bounds = rasterio.transform.array_bounds(height, width, src_transform)
    resolution = (target_resolution, target_resolution) if target_resolution else None
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs, dst_crs, width, height, *bounds, resolution=resolution
    )
    dst_array = np.full((dst_height, dst_width), np.nan, dtype=np.float32)
    reproject(
        source=array, destination=dst_array,
        src_transform=src_transform, src_crs=src_crs, src_nodata=np.nan,
        dst_transform=dst_transform, dst_crs=dst_crs, dst_nodata=np.nan,
        resampling=resampling,
    )
    return dst_array, dst_transform, dst_width, dst_height


def reproject_to_existing_grid(src_fn, dst_transform, dst_width, dst_height, dst_crs, resampling):
    """Reproject a raster FILE onto an already-defined target grid."""
    with rasterio.open(src_fn) as src:
        dst_array = np.full((dst_height, dst_width), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1), destination=dst_array,
            src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
            dst_transform=dst_transform, dst_crs=dst_crs, dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst_array


def load_manning_native(manning_fn):
    """Load an existing manning raster as-is, in its own native CRS/resolution."""
    with rasterio.open(manning_fn) as src:
        array = src.read(1, masked=True).filled(np.nan).astype(np.float32)
        transform = src.transform
        crs = src.crs
    return array, transform, crs

### Auto-pick a fine enough target resolution

Measure the narrowest river-bed segment width and use a fraction of it, so the master grid can never miss a thin reach.

In [ ]:
_gdf_width_check = gpd.read_file(BUFFER_NETWORK_SHP).to_crs(TARGET_CRS)
_gdf_width_check["approx_width_m"] = _gdf_width_check.geometry.area / _gdf_width_check.geometry.length
min_width_m = _gdf_width_check["approx_width_m"].min()
TARGET_RESOLUTION_M = round(min_width_m / 5, 1) #you can change this

print(_gdf_width_check["approx_width_m"].describe())
print(f"\nNarrowest reach approx width: {min_width_m:.1f} m")
print(f"Auto-picked TARGET_RESOLUTION_M = {TARGET_RESOLUTION_M} m (narrowest width / 5)")

In [ ]:
baseline_manning_native, native_transform, native_crs = load_manning_native(EXISTING_MANNING_TIF)

baseline_manning, grid_transform, grid_width, grid_height = reproject_array_to_new_crs(
    baseline_manning_native, native_transform, native_crs, TARGET_CRS,
    resampling=Resampling.nearest,   # keep real class n values -- never interpolate
    target_resolution=TARGET_RESOLUTION_M,
)

print(f"Master grid: {grid_width} x {grid_height} px, CRS={TARGET_CRS}")
print(f"Pixel size: {grid_transform.a:.3f} x {abs(grid_transform.e):.3f} m")
print(f"Baseline manning range: {np.nanmin(baseline_manning):.4f} - {np.nanmax(baseline_manning):.4f}")

## Step 2 — river branches (single fixed value, binary mask)

In [ ]:
branch_mask_arr = reproject_to_existing_grid(
    BRANCH_NETWORK_TIF, grid_transform, grid_width, grid_height, TARGET_CRS,
    resampling=Resampling.nearest  # binary mask -- never interpolate
)
branch_mask = ~np.isnan(branch_mask_arr) & (branch_mask_arr > 0)
print(f"Branch cells: {int(branch_mask.sum())} of {branch_mask.size}")

## Step 3 — main river bed (river_bed_polygons.shp, manning_n field, highest priority)

In [ ]:
gdf = gpd.read_file(BUFFER_NETWORK_SHP).to_crs(TARGET_CRS)
print(f"Loaded {len(gdf)} river bed segments")

### Step 3a — close small coverage gaps before rasterizing

Fixes the bug where `BRANCH_MANNING_VALUE` (0.033) was peaking through tiny gaps/slivers between adjacent digitized river-bed segments.

In [ ]:
# Close small gaps/slivers between adjacent segments so the flat branch value
# (BRANCH_MANNING_VALUE) can never "peak through" inside the main river bed.
# Each segment is grown slightly on its own -- this does NOT dissolve them into
# one shape, so every segment keeps its own manning_n value for rasterizing.
gdf["geometry"] = gdf.geometry.buffer(GAP_CLOSE_BUFFER_M)

area_before = gpd.read_file(BUFFER_NETWORK_SHP).to_crs(TARGET_CRS).geometry.area.sum()
area_after = gdf.geometry.area.sum()
pct_growth = 100 * (area_after - area_before) / area_before
print(f"Coverage area grew by {pct_growth:.2f}% after {GAP_CLOSE_BUFFER_M} m gap-closing buffer")
print("(a large jump here, e.g. >20%, may mean the buffer distance is too big -- reduce GAP_CLOSE_BUFFER_M)")

### Step 3b — rasterize the (now gap-free) river bed segments

In [ ]:
river_bed_arr = rasterize(
    [(geom, val) for geom, val in zip(gdf.geometry, gdf[BUFFER_MANNING_FIELD])],
    out_shape=(grid_height, grid_width),
    transform=grid_transform,
    fill=np.nan,
    dtype="float32",
)
river_bed_mask = ~np.isnan(river_bed_arr)
print(f"Main river bed cells: {int(river_bed_mask.sum())} of {river_bed_arr.size}")
print(f"manning_n value range: {gdf[BUFFER_MANNING_FIELD].min():.4f} - {gdf[BUFFER_MANNING_FIELD].max():.4f}")

## Step 4 — merge (priority: main river bed > branches > baseline manning base) and save

In [ ]:
merged = baseline_manning.copy()
merged[branch_mask] = BRANCH_MANNING_VALUE
merged[river_bed_mask] = river_bed_arr[river_bed_mask]

os.makedirs(os.path.dirname(OUTPUT_MANNING_TIF), exist_ok=True)

profile = {
    "driver": "GTiff", "height": grid_height, "width": grid_width,
    "count": 1, "dtype": "float32", "crs": TARGET_CRS, "transform": grid_transform,
    "nodata": np.nan, "compress": "deflate", "predictor": 2, "tiled": True,
    "blockxsize": 256, "blockysize": 256,
}
with rasterio.open(OUTPUT_MANNING_TIF, "w", **profile) as dst:
    dst.write(merged.astype(np.float32), 1)

print(f"Saved final manning map: {OUTPUT_MANNING_TIF}")
print(f"Final manning range: {np.nanmin(merged):.4f} - {np.nanmax(merged):.4f}")
print(f"Valid (non-NoData) cells: {(~np.isnan(merged)).sum()} of {merged.size}")

### Step 4b — check that only real n values were written

In [ ]:
# Every output cell must carry an n value that exists in the input raster (or a river value).
allowed = set(np.unique(np.round(baseline_manning_native[~np.isnan(baseline_manning_native)], 6)).tolist())
allowed |= {round(float(BRANCH_MANNING_VALUE), 6)}
allowed |= {round(float(v), 6) for v in gdf[BUFFER_MANNING_FIELD].unique()}

present = np.unique(np.round(merged[~np.isnan(merged)], 6))
foreign = sorted(set(present.tolist()) - allowed)
print(f"Distinct n values in output: {len(present)}")
if len(present) <= 50:
    print(present)
if foreign:
    print(f"!! {len(foreign)} value(s) not present in the input raster / river values: {foreign[:20]}")
else:
    print("OK -- every cell holds a value taken directly from the input (no interpolation).")

## Sanity checks (optional — recommended after every run)

- Load `OUTPUT_MANNING_TIF` in QGIS and visually confirm the river bed and branches
  stand out from the surrounding baseline values.
- Confirm the printed pixel size looks reasonable given `TARGET_RESOLUTION_M`.
- Check `river_bed_mask.sum()` and `branch_mask.sum()` aren't zero.
- Click a point on the river centerline and confirm it matches `river_bed_polygons.shp`'s
  `manning_n` value exactly 